# EuroSAT land-use classification with a CNN

EuroSAT: 27 000 Sentinel-2 RGB patches (64×64 px), 10 classes – forest, river, highway,
industrial, residential, annual crop, etc.

A small CNN trained from scratch, with data augmentation. Run on Google Colab (GPU).

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

tf.random.set_seed(42)

In [ ]:
# zip next to the notebook (on Colab: upload it to /content)
ZIP_PATH = Path("Group A -Eurosat Classification.zip")
DATA_DIR = Path("eurosat_data/2750")   # one sub-folder per class

if not DATA_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("eurosat_data")

IMG = 64
BATCH = 64

# validation_split + same seed keeps the two subsets disjoint
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="both", seed=42,
    image_size=(IMG, IMG), batch_size=BATCH)

class_names = train_ds.class_names
print(class_names)

In [ ]:
plt.figure(figsize=(10, 4))
for imgs, labels in train_ds.take(1):
    for i in range(10):
        plt.subplot(2, 5, i + 1)
        plt.imshow(imgs[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]], fontsize=8)
        plt.axis("off")
plt.show()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## Model

Augmentation lives inside the model so it's only active during training (and isn't frozen by `.cache()`).
Satellite images have no "up", so vertical flips and 90° rotations are all valid.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.1),
])

model = keras.Sequential([
    layers.Input(shape=(IMG, IMG, 3)),
    augment,
    layers.Rescaling(1 / 255),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(len(class_names), activation="softmax"),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=40, callbacks=callbacks)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, key in zip(ax, ["accuracy", "loss"]):
    a.plot(history.history[key], label="train")
    a.plot(history.history["val_" + key], label="val")
    a.set_title(key); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
plt.show()

## Evaluation

In [ ]:
y_true = np.concatenate([y for _, y in val_ds])
y_pred = model.predict(val_ds, verbose=0).argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=class_names).plot(
    ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Confusion matrix (validation)")
plt.tight_layout(); plt.show()

Most confusion is between visually similar classes – highway vs. river (both thin lines),
permanent crop vs. herbaceous vegetation / pasture. Next step would be transfer learning
(e.g. a pretrained ResNet/EfficientNet), which typically pushes EuroSAT accuracy above 95 %.